# <b> 02일차 Seaborn으로 심혈관 패턴 보기</b>

- 목표: `seaborn`으로 범주형 비교와 분포 차이를 더 쉽게 시각화합니다.
- 연결: 메인 실습의 countplot, boxplot, heatmap을 보기 전에 읽는 브리지입니다.


In [ ]:
# Shared lecture path bootstrap
from pathlib import Path
from collections import deque
import os
import sys
import unicodedata

try:
    from google.colab import drive  # type: ignore
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
except ImportError:
    pass

SEARCH_ROOTS = (
    Path("/content/drive/MyDrive/Colab Notebooks"),
    Path("/content/drive/MyDrive"),
    Path("/content/drive/Shareddrives"),
    Path.home(),
)
PROJECT_NAMES = {"의료_이미지_2026", "의료_이미지 2026", "앨리스 - 의료이미지 (26.04)"}

def _norm(text: str) -> str:
    return unicodedata.normalize("NFC", text)

def _is_project_root(path: Path) -> bool:
    return (path / "00_shared" / "utils" / "lecture_paths.py").exists()

def _find_project_root() -> Path:
    override = os.environ.get("PROJECT_ROOT")
    if override:
        root = Path(override).expanduser().resolve()
        if _is_project_root(root):
            return root
        raise FileNotFoundError("환경변수 PROJECT_ROOT가 올바른 강의 루트를 가리키지 않습니다.")

    cwd = Path.cwd().resolve()
    for root in [cwd, *cwd.parents]:
        if _is_project_root(root):
            return root

    target_names = {_norm(name) for name in PROJECT_NAMES}
    for base in SEARCH_ROOTS:
        if not base.exists():
            continue
        queue = deque([(base.resolve(), 0)])
        seen = set()
        while queue:
            current, depth = queue.popleft()
            if current in seen:
                continue
            seen.add(current)

            if _norm(current.name) in target_names and _is_project_root(current):
                return current
            if depth >= 4:
                continue

            try:
                children = [
                    child for child in current.iterdir()
                    if child.is_dir() and not child.name.startswith(".") and child.name not in {"__pycache__", ".ipynb_checkpoints"}
                ]
            except OSError:
                continue

            children.sort(key=lambda child: (_norm(child.name) not in target_names, child.name))
            queue.extend((child.resolve(), depth + 1) for child in children)

    raise FileNotFoundError(
        "PROJECT_ROOT를 찾지 못했습니다. Colab에서는 os.environ['PROJECT_ROOT']를 먼저 지정하세요."
    )

def _resolve_day_dir(name: str) -> Path:
    for candidate in {name, unicodedata.normalize("NFC", name), unicodedata.normalize("NFD", name)}:
        path = PROJECT_ROOT / candidate
        if path.exists():
            return path
    return PROJECT_ROOT / name

PROJECT_ROOT = _find_project_root()
os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)
UTILS_ROOT = PROJECT_ROOT / "00_shared" / "utils"
if str(UTILS_ROOT) not in sys.path:
    sys.path.append(str(UTILS_ROOT))

from lecture_paths import DATA_ROOT, MODEL_ROOT, ensure_dir

DAY_DIR = _resolve_day_dir('02일차_의료데이터_탐색_기초실습')
DAY_OUTPUT_ROOT = ensure_dir(DAY_DIR / "99_실행산출물")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DATA_ROOT: {DATA_ROOT}")
print(f"MODEL_ROOT: {MODEL_ROOT}")
print(f"DAY_OUTPUT_ROOT: {DAY_OUTPUT_ROOT}")


## 실행 가이드

- `seaborn`은 `matplotlib` 위에서 더 읽기 쉬운 그래프를 빠르게 그릴 때 유용합니다.
- 한 번에 모든 그래프를 외우기보다 `countplot`, `boxplot`, `heatmap`만 익히는 것을 목표로 합니다.


## AI 도구 활용 체크포인트

- `countplot`과 `boxplot`이 각각 어떤 질문에 답하는 그래프인지 설명하게 해보세요.
- heatmap을 보고 상관이 높아 보이는 변수 쌍을 먼저 후보로 뽑게 한 뒤, 숫자를 직접 확인합니다.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

csv_path = DATA_ROOT / "heart_disease" / "heart_disease.csv"
df = pd.read_csv(csv_path, keep_default_na=False, na_values=[""])
print(f"shape: {df.shape}")

# Part 1. 범주형 비교

## 🎯 학습 목표
- `countplot`으로 범주형 분포를 비교한다.
- `hue`를 사용해 두 집단을 함께 본다.


In [ ]:
plt.figure(figsize=(5, 4))
sns.countplot(x="Heart Disease Status", data=df, palette="pastel")
plt.title("Heart Disease Status Count")
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x="Gender", hue="Heart Disease Status", data=df, palette="Set2")
plt.title("Gender by Heart Disease Status")
plt.show()

# Part 2. 수치형 분포와 상관관계

## 🎯 학습 목표
- `boxplot`으로 집단별 분포 차이를 본다.
- `heatmap`으로 수치형 변수 간 상관관계를 확인한다.


In [ ]:
plt.figure(figsize=(6, 4))
sns.boxplot(x="Heart Disease Status", y="Cholesterol Level", data=df, palette="Set3")
plt.title("Cholesterol by Heart Disease Status")
plt.show()

In [ ]:
corr_cols = ["Age", "Blood Pressure", "Cholesterol Level", "BMI", "Sleep Hours", "Triglyceride Level"]
plt.figure(figsize=(7, 5))
sns.heatmap(df[corr_cols].corr(numeric_only=True), annot=True, cmap="Blues", fmt=".2f")
plt.title("Numeric Feature Correlation")
plt.show()

## 📝 TODO

In [ ]:
# Age 컬럼을 사용해 심장병 유무별 분포를 seaborn 그래프로 하나 더 그려보세요. 예: boxplot 또는 histplot

## 의료 해석 질문

- 성별과 심장병 유무를 함께 그려보면 어떤 차이가 먼저 눈에 띄나요?
- heatmap에서 함께 볼 가치가 있어 보이는 수치형 변수 쌍은 무엇인가요?
